In [1]:
import hashlib

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titleweight"] = "bold"

try:
    con.close()
except:
    pass

con = duckdb.connect("../Data/processed/project.duckdb")

1. WHY ARE THE PARTICIPANTS GROWING SO MUCH OVER TIME? 

In [2]:
#Is it just population growth, or is per-person behavior also intensifying?
con.execute("""
    SELECT campaignId, 
           COUNT(*) AS n_trades, 
           COUNT(DISTINCT accountId) AS n_accounts,
           ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT accountId), 1) AS avg_trades_per_account
    FROM trades
    GROUP BY campaignId
    ORDER BY campaignId
""").fetchdf()

,campaignId,n_trades,n_accounts,avg_trades_per_account
0,33,749,137,5.5
1,34,942,192,4.9
2,35,1030,185,5.6
3,36,1457,221,6.6
4,37,1432,236,6.1
5,38,1271,205,6.2
6,39,1180,205,5.8
7,40,1432,248,5.8
8,41,1227,177,6.9
9,42,1674,269,6.2


because the avg trades per person is same, its just more people participating 

In [3]:
#Are the same accounts showing up across multiple campaigns (repeat participation)?
con.execute("""
    SELECT accountId, COUNT(DISTINCT campaignId) AS num_campaigns_participated
    FROM trades
    GROUP BY accountId
    ORDER BY num_campaigns_participated DESC
""").fetchdf()

,accountId,num_campaigns_participated
0,D#1702721,26
1,D#1702533,26
2,D#1671069,25
3,D#1670945,25
4,D#1671057,24
...,...,...
497,D#1702473,8
498,D#1702707,7
499,D#1645625,1
500,D#1645639,1


We have repeat participation

2. WHY IS THE RATIO BW NO. OF REGISTRY ROWS AND NO. OF UNIQUE ACCOUNTS SO HIGH? 

In [ ]:
# Are the 7,000 rows actual exact duplicates, or genuinely different rows per account?


con.execute("""
    SELECT account, COUNT(*) AS num_rows
    FROM users
    GROUP BY account
    ORDER BY num_rows DESC
    LIMIT 10
""").fetchdf()

,account,num_rows
0,D#1589428,14
1,D#1614623,14
2,D#1670864,14
3,D#1671093,14
4,D#1702338,14
5,D#1702356,14
6,D#1702363,14
7,D#1702369,14
8,D#1702380,14
9,D#1702385,14


That's not a coincidence — 14 is a suspiciously round, uniform number, showing up for every single account in the top 10. That rules out "duplicates from a messy export" (which would usually give uneven counts) and points instead to a structural reason: this registry likely logs one row per account per something that happens 14 times — very possibly once per campaign type/cohort, or a repeated snapshot/log entry taken on a schedule.

In [ ]:
#That's not a coincidence — 14 is a suspiciously round, uniform number, showing up for every single account in the top 10. That rules out "duplicates from a messy export" (which would usually give uneven counts) and points instead to a structural reason: this registry likely logs one row per account per something that happens 14 times — very possibly once per campaign type/cohort, or a repeated snapshot/log entry taken on a schedule.
con.execute("""
    SELECT * FROM users WHERE account = 'D#1589428'
""").fetchdf()

,account,challenge_type_id,email,telegram_username,ip_address
0,D#1589428,11,anothimarian@gmail.com,Marian,3.68.137.171
1,D#1589428,11,mayanksingh9836@gmail.com,@Lotteryteam82,47.128.162.4
2,D#1589428,11,ttarak117@gmail.com,@Tarekgasri,63.178.57.248
3,D#1589428,11,tsan0034@gmail.com,@kenyuyu1,182.63.112.144
4,D#1589428,11,marakben07@gmail.com,Lese,157.42.21.155
5,D#1589428,11,magdumaditya250@gmail.com,Adi magdum,1.39.169.154
6,D#1589428,11,sunilmacker3@gmail.com,@asmaticsunil,3.1.216.214
7,D#1589428,11,marwarazan91@gmail.com,@ran100xx,59.92.132.74
8,D#1589428,11,domcul3@gmail.com,Domcul001,18.156.196.34
9,D#1589428,11,owaishasan766@gmail.com,Owais hasan,54.254.229.208


odd.. are the duplicates? maybe same account diff people? 

In [ ]:
#Does this same pattern hold broadly, or is it isolated?
con.execute("""
    SELECT AVG(distinct_emails) AS avg_emails_per_account,
           AVG(distinct_ips) AS avg_ips_per_account,
           MIN(distinct_emails) AS min_emails,
           MAX(distinct_emails) AS max_emails
    FROM (
        SELECT account,
               COUNT(DISTINCT email) AS distinct_emails,
               COUNT(DISTINCT ip_address) AS distinct_ips
        FROM users
        GROUP BY account
    )
""").fetchdf()

,avg_emails_per_account,avg_ips_per_account,min_emails,max_emails
0,13.2,12.774,10,14


In [ ]:
con.execute("""
    SELECT email, COUNT(DISTINCT account) AS num_accounts
    FROM users
    GROUP BY email
    HAVING COUNT(DISTINCT account) > 1
    ORDER BY num_accounts DESC
""").fetchdf()

,email,num_accounts
0,NaN,289
1,kumaran18101999@gmail.com,14
2,fmwangi748@gmail.com,13
3,ashutoshsontakke2005@gmail.com,13
4,tmokgosi430@gmail.com,13
...,...,...
1056,subham3257@gmail.com,2
1057,ih7117551@gmail.com,2
1058,newtiktokyabu@gmail.com,2
1059,ezramedhine@gmail.com,2


289 accounts share the value "NaN" for email, meaning it's missing/null, not a real shared identity.

that means over half the registry is missing a key indentity field


In [ ]:
con.execute("""
    SELECT email, COUNT(DISTINCT account) AS num_accounts
    FROM users
    WHERE email IS NOT NULL
    GROUP BY email
    HAVING COUNT(DISTINCT account) > 1
    ORDER BY num_accounts DESC
""").fetchdf()



,email,num_accounts
0,kumaran18101999@gmail.com,14
1,fmwangi748@gmail.com,13
2,tmokgosi430@gmail.com,13
3,ashutoshsontakke2005@gmail.com,13
4,kufafx@gmail.com,13
...,...,...
1055,udittrading1994@gmail.com,2
1056,stefanstojanov1375@gmail.com,2
1057,deluixy007@gmail.com,2
1058,habibbhs85@gmail.com,2


but this shows that there many accounts with the same emails 

In [ ]:
#does ip adress have the same completeness problem ( like some email were nan)
con.execute("""
    SELECT COUNT(*) AS total_rows,
           SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS null_emails,
           SUM(CASE WHEN ip_address IS NULL THEN 1 ELSE 0 END) AS null_ips,
           SUM(CASE WHEN telegram_username IS NULL THEN 1 ELSE 0 END) AS null_telegrams
    FROM users
""").fetchdf()

,total_rows,null_emails,null_ips,null_telegrams
0,7000,391.0,391.0,392.0


In [ ]:
# whats the scale of this

con.execute("""
    SELECT COUNT(*) AS num_shared_ips, SUM(num_accounts) AS accounts_involved
    FROM (
        SELECT ip_address, COUNT(DISTINCT account) AS num_accounts
        FROM users
        WHERE ip_address IS NOT NULL
        GROUP BY ip_address
        HAVING COUNT(DISTINCT account) > 1
    )
""").fetchdf()

,num_shared_ips,accounts_involved
0,652,4670.0


In [ ]:
#confirm the trading-account vs. registry-account mismatch
con.execute("""
    SELECT COUNT(DISTINCT t.accountId) AS traded_but_not_in_registry
    FROM trades t
    LEFT JOIN users u ON t.accountId = u.account
    WHERE u.account IS NULL
""").fetchdf()

,traded_but_not_in_registry
0,3


In [ ]:
# account ID of the 3 accounts that traded but have no registry
con.execute("""
    SELECT DISTINCT t.accountId
    FROM trades t
    LEFT JOIN users u ON t.accountId = u.account
    WHERE u.account IS NULL
""").fetchdf()

,accountId
0,D#1645625
1,D#1759507
2,D#1645639


In [ ]:

con.execute("SELECT * FROM users").fetchdf()

,account,challenge_type_id,email,telegram_username,ip_address
0,D#1589378,11,annw57382@gmail.com,Anna,3.126.24.127
1,D#1589383,11,happyked68@gmail.com,@micup235,3.124.228.43
2,D#1589404,11,oyeniranayomikun03@gmail.com,@PRIME_ISAGII,52.58.179.195
3,D#1589405,11,salvationameh66@gmail.com,Salvation,52.29.234.158
4,D#1589407,11,NaN,NaN,NaN
...,...,...,...,...,...
6995,D#1702745,11,newtiktokyabu@gmail.com,https://t.me/Yeaaaaaaaaaaaaaaaaaaaaa,63.178.57.248
6996,D#1702746,11,NaN,NaN,NaN
6997,D#1702748,11,aniqbhai900@gmail.com,Muhammad Shariq,111.92.142.91
6998,D#1702751,11,jangirh185@gmail.com,Harrylake64243,52.221.51.176


In [ ]:
con.execute("DESCRIBE users").fetchdf()

,column_name,column_type,null,key,default,extra
0,account,VARCHAR,YES,None,None,None
1,challenge_type_id,BIGINT,YES,None,None,None
2,email,VARCHAR,YES,None,None,None
3,telegram_username,VARCHAR,YES,None,None,None
4,ip_address,VARCHAR,YES,None,None,None


    3. WHY IS THERE A SPIKE AT THE 150 MARK?

In [4]:
#probing why
con.execute("""
    SELECT n_trades, COUNT(*) AS num_accounts
    FROM (SELECT accountId, COUNT(*) AS n_trades FROM trades_clean GROUP BY accountId)
    WHERE n_trades BETWEEN 140 AND 155
    GROUP BY n_trades
    ORDER BY n_trades
""").fetchdf()

CatalogException: Catalog Error: Table with name trades_clean does not exist!
Did you mean "trades"?

LINE 3:     FROM (SELECT accountId, COUNT(*) AS n_trades FROM trades_clean GROUP BY accountId)
                                                              ^